In [ ]:
from notion_client import Client
import os, json, time, re

NOTION_TOKEN = os.getenv("NOTION_TOKEN")
notion = Client(auth=NOTION_TOKEN)

In [3]:
def search_everything():
    results = []
    has_more = True
    start_cursor = None
    while has_more:
        resp = notion.search(start_cursor=start_cursor, page_size=100)
        results.extend(resp["results"])
        has_more = resp["has_more"]
        start_cursor = resp["next_cursor"]
    return results

everything = search_everything()
page_ids = [p["id"] for p in everything if p["object"] == "page"]
db_ids   = [p["id"] for p in everything if p["object"] == "database"]

In [9]:
everything

[{'object': 'page',
  'id': '258ef339-9239-80d1-be82-d5394d285804',
  'created_time': '2025-08-23T16:21:00.000Z',
  'last_edited_time': '2025-08-24T13:54:00.000Z',
  'created_by': {'object': 'user',
   'id': '258d872b-594c-8105-9b37-0002756b173d'},
  'last_edited_by': {'object': 'user',
   'id': '258d872b-594c-8105-9b37-0002756b173d'},
  'cover': None,
  'icon': None,
  'parent': {'type': 'workspace', 'workspace': True},
  'archived': False,
  'in_trash': False,
  'properties': {'title': {'id': 'title',
    'type': 'title',
    'title': [{'type': 'text',
      'text': {'content': 'RAG Source', 'link': None},
      'annotations': {'bold': False,
       'italic': False,
       'strikethrough': False,
       'underline': False,
       'code': False,
       'color': 'default'},
      'plain_text': 'RAG Source',
      'href': None}]}},
  'url': 'https://www.notion.so/RAG-Source-258ef339923980d1be82d5394d285804',
  'public_url': None},
 {'object': 'page',
  'id': '258ef339-9239-802e-8c3c-c10

In [4]:
def extract_blocks(block_id):
    blocks = []
    has_more = True
    start_cursor = None
    while has_more:
        resp = notion.blocks.children.list(block_id=block_id,
                                           start_cursor=start_cursor,
                                           page_size=100)
        for block in resp["results"]:
            blocks.append(block)
            if block["has_children"]:
                # recurse
                children = extract_blocks(block["id"])
                block["children"] = children
        has_more = resp["has_more"]
        start_cursor = resp["next_cursor"]
        time.sleep(0.35)   # respect rate limit
    return blocks

In [5]:
def block_to_text(block):
    t = block["type"]
    if t in ("paragraph", "heading_1", "heading_2", "heading_3",
             "callout", "quote", "bulleted_list_item",
             "numbered_list_item", "to_do", "toggle", "code"):
        rich_text = block[t].get("rich_text", [])
        return "".join([rt["plain_text"] for rt in rich_text])
    if t == "image":
        return block["image"]["file"]["url"]
    return ""

In [6]:
docs = []
for pid in page_ids:
    title_prop = notion.pages.retrieve(pid)["properties"].get("title", {})
    title = "".join([t["plain_text"] for t in title_prop.get("title", [])])
    blocks = extract_blocks(pid)
    body = "\n".join(block_to_text(b) for b in blocks)
    docs.append({"id": pid, "title": title, "body": body})

In [7]:
docs

[{'id': '258ef339-9239-80d1-be82-d5394d285804',
  'title': 'RAG Source',
  'body': '\n\n\n\n'},
 {'id': '258ef339-9239-802e-8c3c-c1052fee7bf3',
  'title': 'Business Ideas Kolhapur City',
  'body': 'Innovative Business Ideas\n1. The Kolhapuri "Tiffin" Experience (Modernized)\nConcept:\xa0A curated, subscription-based (or a-la-carte) tiffin service that delivers authentic Kolhapuri thalis in modern, eco-friendly packaging. Instead of the standard daily lunch, focus on specialty days.\nMinimal Investment:\xa0Start from your home kitchen. Invest in good quality, branded packaging.\nCultural Twist:\nHow to Innovate:\xa0Offer a "mini-thali" option for office-goers. Partner with local cafes to be their specialty food provider.\n2. Kolhapur Craft & Saree Digital Concierge\nConcept:\xa0A service that connects buyers worldwide with authentic Kolhapur handicrafts (like Kolhapuri chappal/sandals, handwoven sarees like Paithani) but with a modern curation and storytelling approach. You act as a fac

In [1]:
from sentence_transformers import SentenceTransformer

/opt/miniconda3/envs/AgenticRAG/lib/python3.13/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [8]:
# pip install sentence-transformers chromadb langchain langchain-community
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# 1. Load your chunks (same as before)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_splits = []
for d in docs:
    for s in text_splitter.split_text(d["body"]):
        all_splits.append(Document(page_content=s,
                                   metadata={"title": d["title"]}))

# 2. Local embedding model (384-d)
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Store in Chroma
vectordb = Chroma.from_documents(all_splits,
                                 embeddings,
                                 persist_directory="./local_chroma_db")

/var/folders/kd/1qpsy3853zb4s_7055x92ln40000gn/T/ipykernel_2071/343126838.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
